# 03 - Embedding Generation (E5-small-v2)

This notebook generates dense vector representations for all domains using the E5-small-v2 sentence transformer.
These 384-dimensional embeddings will serve as input features for the student distillation model.

**Approach:**
- Combine domain name + title + description into a single text input
- Use the E5-small-v2 model (33M params, fast inference) to produce embeddings
- Generate embeddings for ALL domains in train/val/test (not just teacher-labeled ones)
- Save as memory-mapped numpy arrays for efficient loading during training

In [1]:
import numpy as np
import pandas as pd
import torch
import time
from pathlib import Path

PROJECT_DIR = Path('.').resolve().parent
DATA_DIR = PROJECT_DIR / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

print(f'Project: {PROJECT_DIR.name}')
print(f'PyTorch: {torch.__version__}')
print(f'MPS available: {torch.backends.mps.is_available()}')
print(f'Device: {"mps" if torch.backends.mps.is_available() else "cpu"}')

Project: IAB_LLM_Distillation_Classification
PyTorch: 2.11.0
MPS available: True
Device: mps


In [2]:
# Disable SSL verification for HuggingFace Hub (corporate proxy intercepts certs)
import httpx
from huggingface_hub.utils._http import set_client_factory, hf_request_event_hook

def no_ssl_client_factory():
    return httpx.Client(
        event_hooks={'request': [hf_request_event_hook]},
        follow_redirects=True,
        timeout=None,
        verify=False,
    )

set_client_factory(no_ssl_client_factory)
print('HuggingFace Hub: SSL verification disabled')

HuggingFace Hub: SSL verification disabled


In [3]:
# Load all splits
train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
val_df = pd.read_parquet(PROCESSED_DIR / 'val.parquet')
test_df = pd.read_parquet(PROCESSED_DIR / 'test.parquet')

print(f'Train: {len(train_df):,} rows, {train_df["domain_clean"].nunique():,} unique domains')
print(f'Val:   {len(val_df):,} rows, {val_df["domain_clean"].nunique():,} unique domains')
print(f'Test:  {len(test_df):,} rows, {test_df["domain_clean"].nunique():,} unique domains')

Train: 78,357 rows, 77,588 unique domains
Val:   9,810 rows, 9,699 unique domains


Test:  9,794 rows, 9,699 unique domains


## Domain Text Preparation

For E5 models, the recommended input format is `"query: <text>"` or `"passage: <text>"`. Since we are encoding
documents (domains) that will later be classified, we use the `"passage:"` prefix.

We deduplicate by `domain_clean` first -- each domain gets exactly one embedding regardless of how many
category rows it appears in. The text is constructed as: `domain | title | description`.

In [4]:
def build_embedding_text(row):
    """Construct the text to embed for a domain."""
    parts = [row['domain_clean']]
    if pd.notna(row.get('title')) and len(str(row['title']).strip()) > 2:
        parts.append(str(row['title']).strip()[:200])
    if pd.notna(row.get('description')) and len(str(row['description']).strip()) > 2:
        parts.append(str(row['description']).strip()[:300])
    return 'passage: ' + ' | '.join(parts)


def deduplicate_domains(df):
    """One row per unique domain, keeping the richest text."""
    df = df.copy()
    df['_text_len'] = df['title'].fillna('').str.len() + df['description'].fillna('').str.len()
    deduped = df.sort_values('_text_len', ascending=False).drop_duplicates(subset='domain_clean', keep='first')
    deduped = deduped.drop(columns=['_text_len']).reset_index(drop=True)
    return deduped


train_domains = deduplicate_domains(train_df)
val_domains = deduplicate_domains(val_df)
test_domains = deduplicate_domains(test_df)

train_domains['embed_text'] = train_domains.apply(build_embedding_text, axis=1)
val_domains['embed_text'] = val_domains.apply(build_embedding_text, axis=1)
test_domains['embed_text'] = test_domains.apply(build_embedding_text, axis=1)

print(f'Domains to embed:')
print(f'  Train: {len(train_domains):,}')
print(f'  Val:   {len(val_domains):,}')
print(f'  Test:  {len(test_domains):,}')
print(f'  Total: {len(train_domains) + len(val_domains) + len(test_domains):,}')
print(f'\nSample embed_text:')
for t in train_domains['embed_text'].head(3).tolist():
    print(f'  {t[:120]}...')

Domains to embed:
  Train: 77,588
  Val:   9,699
  Test:  9,699
  Total: 96,986

Sample embed_text:
  passage: balajieduservice.net.in | md ms mbbs admission in india 2022: expert advice | balajieduservice provide info abo...
  passage: superiorrvparts.com | superior rv parts | surplus rv parts depot | 79936 el paso 60629 chicago 11368rv parts ac...
  passage: rvparts-us.com | rv parts-us on-line parts store nationwide | rv parts accessories rv parts sales rv parts acce...


In [ ]:
# Load E5-small-v2
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/e5-small-v2')
embedding_dim = model.get_embedding_dimension()
print(f'Model: intfloat/e5-small-v2')
print(f'Embedding dimension: {embedding_dim}')
print(f'Max sequence length: {model.max_seq_length}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Generate Embeddings

We encode in batches using the sentence-transformers `.encode()` method which handles batching and
device placement automatically. On Apple Silicon (MPS), this should be significantly faster than CPU.

In [6]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
BATCH_SIZE = 256

def encode_domains(texts, split_name):
    """Encode a list of texts and return numpy array of embeddings."""
    print(f'Encoding {split_name}: {len(texts):,} domains (batch_size={BATCH_SIZE}, device={device})')
    start = time.time()
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        device=device,
        normalize_embeddings=True,
    )
    elapsed = time.time() - start
    print(f'  Done in {elapsed:.1f}s ({len(texts)/elapsed:.0f} domains/sec)')
    print(f'  Shape: {embeddings.shape}, dtype: {embeddings.dtype}')
    return embeddings


train_embeddings = encode_domains(train_domains['embed_text'].tolist(), 'train')
val_embeddings = encode_domains(val_domains['embed_text'].tolist(), 'val')
test_embeddings = encode_domains(test_domains['embed_text'].tolist(), 'test')

Encoding train: 77,588 domains (batch_size=256, device=mps)


Batches:   0%|          | 0/304 [00:00<?, ?it/s]

  Done in 124.6s (623 domains/sec)
  Shape: (77588, 384), dtype: float32
Encoding val: 9,699 domains (batch_size=256, device=mps)


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

  Done in 16.5s (587 domains/sec)
  Shape: (9699, 384), dtype: float32
Encoding test: 9,699 domains (batch_size=256, device=mps)


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

  Done in 18.7s (518 domains/sec)
  Shape: (9699, 384), dtype: float32


In [7]:
# Quick sanity check: embeddings are normalized (L2 norm ~= 1.0)
norms = np.linalg.norm(train_embeddings[:100], axis=1)
print(f'L2 norm stats (first 100 train embeddings):')
print(f'  Mean: {norms.mean():.6f}')
print(f'  Std:  {norms.std():.6f}')
print(f'  Min:  {norms.min():.6f}')
print(f'  Max:  {norms.max():.6f}')

# Check similarity between a few domains
print(f'\nCosine similarity spot-check (normalized vectors => dot product):')
for i in range(3):
    for j in range(i+1, min(i+3, 5)):
        sim = np.dot(train_embeddings[i], train_embeddings[j])
        print(f'  {train_domains.iloc[i]["domain_clean"]} vs {train_domains.iloc[j]["domain_clean"]}: {sim:.4f}')

L2 norm stats (first 100 train embeddings):
  Mean: 1.000000
  Std:  0.000000
  Min:  1.000000
  Max:  1.000000

Cosine similarity spot-check (normalized vectors => dot product):
  balajieduservice.net.in vs superiorrvparts.com: 0.8109
  balajieduservice.net.in vs rvparts-us.com: 0.8393
  superiorrvparts.com vs rvparts-us.com: 0.9293
  superiorrvparts.com vs waterdamage-us.com: 0.8482
  rvparts-us.com vs waterdamage-us.com: 0.8444
  rvparts-us.com vs superiorrvpartscountyus.com: 0.9195


In [8]:
# Save embeddings and domain indices
EMBEDDINGS_DIR = PROCESSED_DIR / 'embeddings'
EMBEDDINGS_DIR.mkdir(exist_ok=True)

np.save(EMBEDDINGS_DIR / 'train_embeddings.npy', train_embeddings)
np.save(EMBEDDINGS_DIR / 'val_embeddings.npy', val_embeddings)
np.save(EMBEDDINGS_DIR / 'test_embeddings.npy', test_embeddings)

# Save domain order (so we can map embedding index -> domain)
train_domains[['domain_clean']].to_parquet(EMBEDDINGS_DIR / 'train_domains.parquet', index=False)
val_domains[['domain_clean']].to_parquet(EMBEDDINGS_DIR / 'val_domains.parquet', index=False)
test_domains[['domain_clean']].to_parquet(EMBEDDINGS_DIR / 'test_domains.parquet', index=False)

# Report file sizes
for f in sorted(EMBEDDINGS_DIR.glob('*')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}: {size_mb:.1f} MB')

print(f'\nTotal embeddings saved: {len(train_embeddings) + len(val_embeddings) + len(test_embeddings):,} domains')
print(f'Embedding dimension: {train_embeddings.shape[1]}')

  test_domains.parquet: 0.2 MB
  test_embeddings.npy: 14.2 MB
  train_domains.parquet: 1.2 MB
  train_embeddings.npy: 113.7 MB
  val_domains.parquet: 0.2 MB
  val_embeddings.npy: 14.2 MB

Total embeddings saved: 96,986 domains
Embedding dimension: 384


## Reasoning: Embedding Generation Results

**Throughput:** E5-small-v2 on MPS (Apple Silicon) encoded 96,986 domains in ~160 seconds total -- roughly 600 domains/sec. This is fast enough that we can regenerate embeddings cheaply if needed.

**Normalization:** All embeddings have L2 norm of exactly 1.0 (std=0), confirming `normalize_embeddings=True` worked correctly. This means cosine similarity = dot product, simplifying downstream distance computations.

**Similarity sanity check:** The model captures semantic structure well:
- `superiorrvparts.com` vs `rvparts-us.com`: 0.93 (same niche, nearly identical content)
- `superiorrvparts.com` vs `balajieduservice.net.in`: 0.81 (unrelated domains, lower but still positive due to shared "service" semantics)

This confirms the embeddings are encoding meaningful content signals beyond just domain name lexical overlap.

**Storage:** Total ~143 MB for all three splits as float32 numpy arrays. This fits comfortably in memory for training.

**Teacher alignment:** All 6,497 teacher-labeled domains map 1:1 to training embeddings (100% coverage). This is the expected result since teacher labeling sampled from the same training set.

In [9]:
# Verify alignment with teacher labels
teacher_labels = pd.read_parquet(PROCESSED_DIR / 'teacher_labels.parquet')
train_domain_list = train_domains['domain_clean'].tolist()
teacher_in_train = teacher_labels[teacher_labels['domain_clean'].isin(train_domain_list)]

# Find indices of teacher-labeled domains in train embeddings
domain_to_idx = {d: i for i, d in enumerate(train_domain_list)}
teacher_indices = [domain_to_idx[d] for d in teacher_in_train['domain_clean'] if d in domain_to_idx]

print(f'Teacher-labeled domains in train embeddings: {len(teacher_indices):,} / {len(teacher_labels):,}')
print(f'Coverage: {len(teacher_indices)/len(teacher_labels)*100:.1f}%')
print(f'\nThese {len(teacher_indices):,} domains have both embeddings AND teacher soft labels.')
print(f'They will form the distillation training set in Notebook 04.')

Teacher-labeled domains in train embeddings: 6,497 / 6,497
Coverage: 100.0%

These 6,497 domains have both embeddings AND teacher soft labels.
They will form the distillation training set in Notebook 04.
